In [1]:
from brian2 import *
sys.path.append('Neuron and Synapse Models')
from neuronModels import *
from ringAttractorClass import *

sys.path.append('Tools')
from plottingTools import *
from utils import *

set_device('cpp_standalone', build_on_run=False)

In [ ]:
mujocoFlag = False

In [3]:
# Simulation parameters
defaultclock.dt = 0.1*ms

In [4]:
# Setting network parameters
num_neurons = 120
tau=10*ms
sigma_noise=0.1*mV
V_rest=-70*mV

In [ ]:
# External input: a spatially modulated current.

# For example, we define a Gaussian input centered at a particular position (stimulus_center)
stimulus_center = 3.14  # center of the bump on the ring
stimulus_width = 0.5  # width in radians
I0 = 30*mV         # amplitude of the external input

# Define the external input as a function of neuron position
def externalInput(stimulus_center, stimulus_width, I0):
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
# I_ext_array = I0 * exp(-((positions - stimulus_center)**2) / (2 * stimulus_width**2))
    d = np.angle(np.exp(1j * (positions - stimulus_center)))
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2))
    return I_ext_array

I_ext_array = externalInput(stimulus_center, stimulus_width, I0)

In [6]:
# Creating the equation object
neuron_eq = Equations(LIF_xi_vel_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)

In [7]:
# # Connectivity parameters - Mexican hat
# sigma_exc = 0.0875
# sigma_inh = 0.25
# g_exc = 1.0*mV
# g_inh = -0.475*mV

# Testing values
sigma_exc_val= 0.125
sigma_inh_val= 0.25
g_exc= 0.875*mV
g_inh= -0.475*mV

# Connectivity parameters - Cosine
g_cosine = 0.1*mV
w_inh = -0.555*mV

# Create neuron group
Vth=-48*mV
V_reset=-80*mV
refractory_period=5*ms
glob_inh_flag = True
            
ringAttractor = RingAttractor(neuron_eq, 
                        num_neurons, 
                        Vth, V_reset, refractory_period,
                        syn_profile='cosine',
                        autapse=True,
                        glob_inh=glob_inh_flag, w_inh=w_inh,
                        g_cosine=g_cosine,
                        sigma_exc=sigma_exc_val, sigma_inh=sigma_inh_val,
                        g_exc=g_exc, g_inh=g_inh)

ringAttractor.ring_pool.I_ext = I_ext_array
ringAttractor.ring_pool.I_vel = 0.0

ringAttractor.ring_pool.run_regularly('V = clip(V, V_reset, inf*volt)', dt=defaultclock.dt)

CodeRunner(clock=Clock(dt=100. * usecond, name='ring_neurons_run_regularly_clock'), when=start, order=0, name='ring_neurons_run_regularly')

In [ ]:
# Setup monitors: spike monitor and state monitor for membrane potential and external input
spikemon = SpikeMonitor(ringAttractor.ring_pool)
statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)  # if you want to check the input

# Set of Brian objects to be added to the network
localObjects = [spikemon, statemon, inputmon] # enforce_lower_bound]

if glob_inh_flag:
    statemon_inh = StateMonitor(ringAttractor.glob_inh_neuron, 'V', record=True)
    spikemon_inh = SpikeMonitor(ringAttractor.glob_inh_neuron)
    localObjects.extend([statemon_inh, spikemon_inh])
    

net = Network(ringAttractor.BrianObjects+localObjects)
net.run()

WARNING    'w_inh' is an internal variable of group 'glob_inh2pool', but also exists in the run namespace with the value -0.555 * mvolt. The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


In [ ]:
device.build(directory = 'standalone_build_mujoco', compile=True, run=False, debug=False)

In [ ]:
while mujocoFlag:
    
    ringAttractor.ring_synapses_asym.vel_in = 0.0
    run_args={vel_in: mujocoVel, v}
    device.run()